# VeriCall — Execution-Verified Post-Training for Tool Calling
### Reproducible Google Colab AI Pro Training & Evaluation Pipeline (L4 24GB / A100 40GB)
**Author:** Bhuvnesh Sahu ([@bhuvneshsahu01](https://github.com/bhuvneshsahu01))

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhuvneshsahu01/vericall/blob/main/VeriCall_Colab_Pro.ipynb)

---
### Overview
VeriCall aligns LLMs to generate 100% schema-compliant, execution-verified tool calls without runtime parser crashes or schema hallucinations.
This notebook runs the complete 10-stage pipeline:
1. **Environment Setup:** Fast 4-bit Unsloth, TRL, FastAPI sandbox, and evaluation harnesses.
2. **Sandbox Initialization:** Launches an in-memory FastAPI sandbox across 12 endpoints.
3. **Model Initialization:** Loads `unsloth/Qwen2.5-7B-Instruct` in 4-bit NF4 quantization.
4. **Baseline Evaluation:** Evaluates unaligned Base Model & Constrained Decoding on $M=500$ frozen unseen API requests.
5. **QLoRA Supervised Fine-Tuning (SFT):** Fine-tunes low-rank adapters on domain-partitioned training data.
6. **SFT Evaluation:** Evaluates SFT adapter on unseen APIs.
7. **Execution-Grounded DPO Mining:** Mines near-miss preference pairs validated against the FastAPI sandbox.
8. **QLoRA Direct Preference Optimization (DPO):** Aligns the model with $\beta=0.1$ on execution feedback.
9. **Rigorous Statistical Evaluation:** Evaluates SFT+DPO, computing 95% bootstrap CIs (10,000 resamples) and paired McNemar tests.
10. **Results Export & LoRA Packaging:** Exports `results/eval_summary.json` and prepares HuggingFace Hub artifact.


### Stage 1: Environment Setup & Dependencies


In [ ]:
# Install Unsloth, TRL, PEFT, and dependencies
!pip install --quiet --upgrade pip
!pip install --quiet "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --quiet --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes
!pip install --quiet fastapi uvicorn httpx pydantic statsmodels scipy scikit-learn tabulate

# Clone VeriCall repository if running standalone
import os
import sys

if not os.path.exists("vericall"):
    !git clone https://github.com/bhuvneshsahu01/vericall.git
    %cd vericall
else:
    %cd vericall

sys.path.insert(0, os.getcwd())
print("[✓] Dependencies installed and repository set up. Working directory:", os.getcwd())


### Stage 2: In-Process FastAPI Mock Sandbox & Validator


In [ ]:
from vericall.sandbox.validator import ToolCallValidator
from vericall.sandbox.app import app

validator = ToolCallValidator()
print(f"[✓] Initialized ToolCallValidator with {len(validator.registry)} endpoints across 4 domains.")

# Smoke test all 12 endpoints
test_endpoints = [
    ("get_stock_quote", {"symbol": "AAPL", "exchange": "NASDAQ"}),
    ("execute_trade", {"account_id": "acc_1001", "symbol": "AAPL", "quantity": 10, "order_type": "market"}),
    ("get_account_balance", {"account_id": "acc_1001"}),
    ("search_products", {"query": "monitor", "category": "electronics"}),
    ("create_order", {"customer_id": "cust_101", "shipping_city": "Austin", "postal_code": "78701", "items": [{"product_id": "prod_1", "quantity": 1}]}),
    ("track_shipment", {"tracking_number": "TRK12345678", "carrier": "fedex"}),
    ("provision_server", {"instance_type": "c5.xlarge", "region": "us-east-1", "count": 1, "environment": "prod"}),
    ("restart_service", {"service_name": "nginx", "cluster_id": "k8s-prod-us-east-1"}),
    ("get_cluster_metrics", {"cluster_id": "k8s-prod-us-east-1", "metric_name": "cpu_utilization", "time_window_minutes": 15}),
    ("search_flights", {"origin_airport": "JFK", "destination_airport": "LHR", "departure_date": "2026-11-01", "passengers": 1, "cabin_class": "economy"}),
    ("book_hotel", {"hotel_id": "htl-101", "checkin_date": "2026-12-01", "checkout_date": "2026-12-05", "guests": 1, "room_type": "standard"}),
    ("cancel_booking", {"booking_id": "bk-12345", "reason": "schedule_change"}),
]

for fn, args in test_endpoints:
    res = validator.validate_call(fn, args)
    assert res.is_success, f"Endpoint {fn} failed: {res.error_message}"

print("[✓] All 12 sandbox endpoints verified functional.")


### Stage 3: Load Model & Tokenizer (4-bit NF4 Quantization)


In [ ]:
import torch
from unsloth import FastLanguageModel

max_seq_length = 2048
model_name = "unsloth/Qwen2.5-7B-Instruct"

print(f"[*] Loading {model_name} in 4-bit...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
)

print(f"[✓] Model loaded. VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")


### Stage 4: Evaluate Base Model on Frozen M=500 Unseen APIs


In [ ]:
import json
from pathlib import Path
from tqdm import tqdm
from vericall.eval.evaluate import evaluate_predictions

# Verify test manifest and load M=500 unseen API test set
unseen_path = Path("data/eval_unseen_apis.jsonl")
test_records = [json.loads(line) for line in open(unseen_path, "r", encoding="utf-8") if line.strip()]
print(f"[*] Loaded {len(test_records)} frozen unseen API evaluation requests.")

TOOL_SYSTEM_PROMPT = """You are a precise, tool-calling assistant. Given the user's request, produce a JSON object representing the exact tool call with the required parameters.

Output format:
{"name": "<function_name>", "arguments": {<key_value_pairs>}}
"""

FastLanguageModel.for_inference(model)

base_predictions = []
print("[*] Generating predictions with Base Model...")
for item in tqdm(test_records):
    prompt = f"<|im_start|>system\n{TOOL_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{item['query']}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True, temperature=0.1)
    response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    try:
        parsed = json.loads(response_text)
        fn = parsed.get("name", "")
        args = parsed.get("arguments", {})
    except Exception:
        fn = "invalid_json"
        args = {}

    base_predictions.append({"name": fn, "arguments": args})

base_results = evaluate_predictions(test_records, base_predictions, "base_model")
print(f"\n[Base Model] Unseen Accuracy: {base_results['accuracy_pct']}% (Passed: {base_results['passed_requests']}/{base_results['total_requests']})")
print(f"             95% Bootstrap CI: [{base_results['bootstrap_95_ci'][0]}%, {base_results['bootstrap_95_ci'][1]}%]")
print(f"             Errors: {base_results['error_taxonomy']}")


### Stage 5: QLoRA Supervised Fine-Tuning (SFT)


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from datasets import Dataset
from vericall.train.train_sft import load_sft_dataset

# Add LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

sft_data = load_sft_dataset()
train_dataset = Dataset.from_list(sft_data)

training_args = TrainingArguments(
    output_dir="checkpoints/runs_sft",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=20,
    optim="adamw_8bit",
    weight_decay=0.01,
    lr_scheduler_type="cosine",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    args=training_args,
)

print("[*] Starting QLoRA SFT training...")
trainer.train()

model.save_pretrained("checkpoints/vericall-sft-lora")
tokenizer.save_pretrained("checkpoints/vericall-sft-lora")
print("[✓] SFT training complete. Adapters saved to checkpoints/vericall-sft-lora.")


### Stage 6: Evaluate SFT Model on M=500 Unseen APIs


In [ ]:
FastLanguageModel.for_inference(model)

sft_predictions = []
print("[*] Generating predictions with SFT Model...")
for item in tqdm(test_records):
    prompt = f"<|im_start|>system\n{TOOL_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{item['query']}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True, temperature=0.1)
    response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    try:
        parsed = json.loads(response_text)
        fn = parsed.get("name", "")
        args = parsed.get("arguments", {})
    except Exception:
        fn = "invalid_json"
        args = {}

    sft_predictions.append({"name": fn, "arguments": args})

sft_results = evaluate_predictions(test_records, sft_predictions, "sft_only")
print(f"\n[SFT Model] Unseen Accuracy: {sft_results['accuracy_pct']}% (Passed: {sft_results['passed_requests']}/{sft_results['total_requests']})")
print(f"            95% Bootstrap CI: [{sft_results['bootstrap_95_ci'][0]}%, {sft_results['bootstrap_95_ci'][1]}%]")
print(f"            Errors: {sft_results['error_taxonomy']}")


### Stage 7: Execution-Grounded DPO Near-Miss Mining


In [ ]:
from vericall.data.mine_dpo import mine_dpo_pairs

dpo_pairs = mine_dpo_pairs(num_pairs=1000)
print(f"[✓] Successfully mined {len(dpo_pairs)} execution-verified preference pairs.")
print("Sample Pair:")
print("  Prompt:  ", dpo_pairs[0]["prompt"])
print("  Chosen:  ", dpo_pairs[0]["chosen"])
print("  Rejected:", dpo_pairs[0]["rejected"], f"({dpo_pairs[0]['rejected_status']})")


### Stage 8: QLoRA Direct Preference Optimization (DPO)


In [ ]:
from unsloth import PatchDPOTrainer
from trl import DPOTrainer, DPOConfig
from vericall.train.train_dpo import load_dpo_dataset

PatchDPOTrainer()

dpo_config = DPOConfig(
    output_dir="checkpoints/runs_dpo",
    beta=0.1,
    learning_rate=5e-5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    optim="adamw_8bit",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    seed=42,
    max_length=max_seq_length,
    max_prompt_length=1024,
)

dpo_pairs_data = load_dpo_dataset()
dpo_train_dataset = Dataset.from_list(dpo_pairs_data)

dpo_trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_config,
    train_dataset=dpo_train_dataset,
    tokenizer=tokenizer,
)

print("[*] Starting QLoRA DPO alignment...")
dpo_trainer.train()

model.save_pretrained("checkpoints/vericall-dpo-lora")
tokenizer.save_pretrained("checkpoints/vericall-dpo-lora")
print("[✓] DPO training complete. Adapters saved to checkpoints/vericall-dpo-lora.")


### Stage 9: Final Evaluation & Statistical Rigor (Bootstrap CI & McNemar Test)


In [ ]:
from vericall.eval.stats import compute_mcnemar_test

FastLanguageModel.for_inference(model)

dpo_predictions = []
print("[*] Generating final predictions with VeriCall (SFT+DPO)...")
for item in tqdm(test_records):
    prompt = f"<|im_start|>system\n{TOOL_SYSTEM_PROMPT}<|im_end|>\n<|im_start|>user\n{item['query']}<|im_end|>\n<|im_start|>assistant\n"
    inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=128, use_cache=True, temperature=0.1)
    response_text = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    try:
        parsed = json.loads(response_text)
        fn = parsed.get("name", "")
        args = parsed.get("arguments", {})
    except Exception:
        fn = "invalid_json"
        args = {}

    dpo_predictions.append({"name": fn, "arguments": args})

dpo_results = evaluate_predictions(test_records, dpo_predictions, "sft_dpo_vericall")

# Compute McNemar tests
stat_base, p_base = compute_mcnemar_test(base_results["binary_scores"], dpo_results["binary_scores"])
stat_sft, p_sft = compute_mcnemar_test(sft_results["binary_scores"], dpo_results["binary_scores"])

print(f"\n=======================================================")
print(f"FINAL VERICALL BENCHMARK RESULTS (M=500 Unseen APIs)")
print(f"=======================================================")
print(f"[Base Model]     Accuracy: {base_results['accuracy_pct']}%  95% CI: [{base_results['bootstrap_95_ci'][0]}%, {base_results['bootstrap_95_ci'][1]}%]")
print(f"[SFT-Only]       Accuracy: {sft_results['accuracy_pct']}%  95% CI: [{sft_results['bootstrap_95_ci'][0]}%, {sft_results['bootstrap_95_ci'][1]}%]")
print(f"[VeriCall (DPO)] Accuracy: {dpo_results['accuracy_pct']}%  95% CI: [{dpo_results['bootstrap_95_ci'][0]}%, {dpo_results['bootstrap_95_ci'][1]}%]")
print(f"\nMcNemar Test vs Base:    chi2 = {stat_base:.4f}, p = {p_base:.4e} (p < 0.05: {p_base < 0.05})")
print(f"McNemar Test vs SFT-Only: chi2 = {stat_sft:.4f}, p = {p_sft:.4e} (p < 0.05: {p_sft < 0.05})")
print(f"=======================================================")


### Stage 10: Save Evaluation Summary & Render Results


In [ ]:
from pathlib import Path
from scripts.render_results import format_markdown_table, format_latex_snippet

summary = {
    "metadata": {
        "model_base": "unsloth/Qwen2.5-7B-Instruct",
        "eval_set": "eval_unseen_apis.jsonl",
        "eval_count": 500,
        "domains_unseen": ["infra", "travel"],
    },
    "configs": {
        "base_model": base_results,
        "sft_only": sft_results,
        "sft_dpo_vericall": dpo_results,
    },
    "mcnemar_tests": {
        "vericall_vs_base": {"statistic": stat_base, "p_value": p_base},
        "vericall_vs_sft": {"statistic": stat_sft, "p_value": p_sft},
    }
}

Path("results").mkdir(exist_ok=True)
with open("results/eval_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2)

print("[✓] Saved results to results/eval_summary.json")
!python scripts/render_results.py
